# Phase 0: Mathematical & Computational Foundations  
## Notebook 00_04 — Optimization Basics with CVXPY

### Research question

How can convex optimization be used to express and solve realistic portfolio construction problems with constraints, regularisation, and transaction-cost penalties?

This notebook introduces the basic optimization workflow used throughout quantitative finance:

1. **Decision variables**
2. **Objective functions**
3. **Constraints**
4. **Convexity and disciplined convex programming**
5. **Minimum-variance portfolio optimization**
6. **Efficient-frontier construction**
7. **Regularisation and turnover penalties**
8. **Dual variables and economic interpretation**
9. **Solver diagnostics and infeasibility checks**

The goal is to move from closed-form formulas to a more realistic modelling workflow where practical investment constraints can be written directly and solved numerically.

## 1. Theory: optimization problems in finance

A generic optimization problem can be written as:

$$
\begin{aligned}
\text{minimise} \quad & f_0(x) \\
\text{subject to} \quad & f_i(x) \leq 0, \quad i = 1,\dots,m \\
& h_j(x) = 0, \quad j = 1,\dots,p
\end{aligned}
$$

where:

- $x$ is the decision variable;
- $f_0(x)$ is the objective function;
- $f_i(x) \leq 0$ are inequality constraints;
- $h_j(x) = 0$ are equality constraints.

In portfolio optimization, $x$ is usually the vector of portfolio weights:

$$
w =
\begin{bmatrix}
w_1 \\
w_2 \\
\vdots \\
w_N
\end{bmatrix}
$$

A classic risk objective is portfolio variance:

$$
w^\top \Sigma w
$$

where $\Sigma$ is the covariance matrix of asset returns.

This is a convex quadratic function when $\Sigma$ is positive semi-definite.

## 2. Why CVXPY?

CVXPY is a Python modelling language for convex optimization.

Instead of manually deriving a custom algorithm for every problem, we express the problem mathematically:

1. define variables;
2. define an objective;
3. define constraints;
4. solve the problem;
5. inspect the solution and diagnostics.

The core modelling pattern is:

```python
w = cp.Variable(n)
objective = cp.Minimize(...)
constraints = [...]
problem = cp.Problem(objective, constraints)
problem.solve()
```

This notebook uses CVXPY because later notebooks will need constrained optimization for:

- mean-variance portfolios;
- risk parity approximations;
- CVaR minimisation;
- factor exposure control;
- transaction-cost-aware rebalancing;
- optimal execution;
- robust portfolio construction.

In [ ]:
# If CVXPY is not installed in your environment, run:
# pip install cvxpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import cvxpy as cp
except ImportError as exc:
    raise ImportError(
        "CVXPY is required for this notebook. Install it with: pip install cvxpy"
    ) from exc

rng = np.random.default_rng(seed=42)

print("CVXPY version:", cp.__version__)

## 3. Synthetic portfolio data

To keep the notebook reproducible, we generate synthetic expected returns and a positive semi-definite covariance matrix.

We simulate a simple factor structure:

$$
r_t = B f_t + \epsilon_t
$$

The true covariance matrix is:

$$
\Sigma = B \Sigma_f B^\top + D
$$

where $D$ is a diagonal idiosyncratic covariance matrix.

This gives realistic cross-asset correlation while still being fully controlled and reproducible.

In [ ]:
def generate_synthetic_portfolio_inputs(
    num_assets: int,
    num_factors: int,
    seed: int = 42
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Generate synthetic expected returns and a positive definite covariance matrix.

    Returns
    -------
    mu:
        Expected returns vector.

    cov:
        Positive definite covariance matrix.

    asset_table:
        Human-readable table of asset names and expected returns.
    """
    local_rng = np.random.default_rng(seed)

    # Factor loadings.
    loadings = local_rng.normal(loc=0.0, scale=0.8, size=(num_assets, num_factors))

    # Factor covariance.
    raw = local_rng.normal(size=(num_factors, num_factors))
    factor_cov = raw @ raw.T

    # Scale factor covariance to daily-return magnitudes.
    factor_cov = factor_cov / np.max(np.diag(factor_cov))
    factor_cov = factor_cov * (0.012 ** 2)

    # Idiosyncratic risk.
    idio_vols = local_rng.uniform(0.006, 0.018, size=num_assets)
    idio_cov = np.diag(idio_vols ** 2)

    cov = loadings @ factor_cov @ loadings.T + idio_cov

    # Synthetic expected returns with a weak relationship to factor exposure.
    base_mu = local_rng.normal(loc=0.00035, scale=0.00015, size=num_assets)
    factor_premium = 0.00008 * loadings[:, 0]
    mu = base_mu + factor_premium

    asset_names = [f"Asset_{i:02d}" for i in range(1, num_assets + 1)]

    asset_table = pd.DataFrame({
        "asset": asset_names,
        "expected_daily_return": mu,
        "daily_volatility": np.sqrt(np.diag(cov))
    })

    return mu, cov, asset_table

In [ ]:
num_assets = 20
num_factors = 4

mu, cov, asset_table = generate_synthetic_portfolio_inputs(
    num_assets=num_assets,
    num_factors=num_factors,
    seed=42
)

asset_table.head()

In [ ]:
def covariance_diagnostics(cov: np.ndarray) -> pd.Series:
    """
    Basic covariance matrix diagnostics.
    """
    eigenvalues = np.linalg.eigvalsh(cov)
    return pd.Series({
        "min_eigenvalue": eigenvalues.min(),
        "max_eigenvalue": eigenvalues.max(),
        "condition_number": eigenvalues.max() / eigenvalues.min(),
        "is_symmetric": np.allclose(cov, cov.T),
        "numerical_rank": np.linalg.matrix_rank(cov)
    })


covariance_diagnostics(cov)

## 4. Problem 1: long-only minimum variance portfolio

The global minimum variance problem is:

$$
\begin{aligned}
\text{minimise} \quad & w^\top \Sigma w \\
\text{subject to} \quad & \mathbf{1}^\top w = 1 \\
& w_i \geq 0 \quad \text{for all } i
\end{aligned}
$$

The long-only constraint prevents short selling.

This is a convex quadratic program because:

- $w^\top \Sigma w$ is convex when $\Sigma$ is positive semi-definite;
- the budget constraint is affine;
- the long-only constraint is affine.

In [ ]:
def solve_long_only_min_variance(cov: np.ndarray) -> dict:
    """
    Solve the long-only global minimum variance portfolio.
    """
    n = cov.shape[0]

    w = cp.Variable(n)

    objective = cp.Minimize(cp.quad_form(w, cov))
    constraints = [
        cp.sum(w) == 1,
        w >= 0
    ]

    problem = cp.Problem(objective, constraints)
    optimal_value = problem.solve()

    return {
        "problem": problem,
        "weights": np.asarray(w.value).flatten(),
        "status": problem.status,
        "optimal_value": optimal_value,
        "budget_dual": constraints[0].dual_value,
        "long_only_dual": constraints[1].dual_value
    }


min_var_result = solve_long_only_min_variance(cov)

print("Status:", min_var_result["status"])
print("Portfolio variance:", min_var_result["optimal_value"])
print("Sum of weights:", min_var_result["weights"].sum())

In [ ]:
min_var_weights = pd.DataFrame({
    "asset": asset_table["asset"],
    "weight": min_var_result["weights"]
})

min_var_weights.sort_values("weight", ascending=False).head(10)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(min_var_weights["asset"], min_var_weights["weight"])
plt.xticks(rotation=90)
plt.title("Long-Only Minimum Variance Portfolio Weights")
plt.xlabel("Asset")
plt.ylabel("Weight")
plt.show()

## 5. Adding a target return constraint

A more demanding problem is:

$$
\begin{aligned}
\text{minimise} \quad & w^\top \Sigma w \\
\text{subject to} \quad & \mathbf{1}^\top w = 1 \\
& \mu^\top w \geq r_{\text{target}} \\
& w_i \geq 0 \quad \text{for all } i
\end{aligned}
$$

This asks for the lowest-risk long-only portfolio that achieves at least a target expected return.

By sweeping over different target returns, we can trace an efficient frontier.

In [ ]:
def solve_target_return_portfolio(
    mu: np.ndarray,
    cov: np.ndarray,
    target_return: float
) -> dict:
    """
    Solve a long-only minimum variance portfolio with a target expected return.
    """
    n = len(mu)

    w = cp.Variable(n)

    objective = cp.Minimize(cp.quad_form(w, cov))
    constraints = [
        cp.sum(w) == 1,
        mu @ w >= target_return,
        w >= 0
    ]

    problem = cp.Problem(objective, constraints)
    optimal_value = problem.solve()

    if w.value is None:
        weights = np.full(n, np.nan)
    else:
        weights = np.asarray(w.value).flatten()

    return {
        "status": problem.status,
        "weights": weights,
        "variance": optimal_value,
        "target_return": target_return,
        "achieved_return": np.nan if w.value is None else float(mu @ weights),
        "target_return_dual": constraints[1].dual_value
    }

In [ ]:
target_grid = np.linspace(mu.min(), mu.max(), 40)

frontier_rows = []

for target in target_grid:
    result = solve_target_return_portfolio(mu, cov, target)
    if result["status"] in ["optimal", "optimal_inaccurate"]:
        frontier_rows.append({
            "target_return": target,
            "achieved_return": result["achieved_return"],
            "volatility": np.sqrt(result["variance"]),
            "variance": result["variance"],
            "target_return_dual": result["target_return_dual"]
        })

frontier_df = pd.DataFrame(frontier_rows)
frontier_df.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(frontier_df["volatility"], frontier_df["achieved_return"], marker="o")
plt.title("Long-Only Efficient Frontier")
plt.xlabel("Portfolio volatility")
plt.ylabel("Expected daily return")
plt.show()

## 6. Dual variables and economic interpretation

CVXPY allows us to inspect **dual variables** associated with constraints.

For the target-return constraint:

$$
\mu^\top w \geq r_{\text{target}}
$$

the dual variable can be interpreted as a shadow price.

Roughly, it measures how much the objective would change if the constraint became slightly tighter.

If the dual value is close to zero, the constraint is probably not binding.

If the dual value is large, the target-return constraint is expensive in terms of additional portfolio variance.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(frontier_df["target_return"], frontier_df["target_return_dual"], marker="o")
plt.title("Dual Value of Target Return Constraint")
plt.xlabel("Target expected daily return")
plt.ylabel("Dual value")
plt.show()

## 7. Regularisation: penalising extreme weights

In practice, unconstrained or weakly constrained optimisers can produce unstable weights.

A common stabilisation technique is to add an $L_2$ penalty:

$$
\lambda \|w\|_2^2
$$

The optimization problem becomes:

$$
\begin{aligned}
\text{minimise} \quad & w^\top \Sigma w + \lambda \|w\|_2^2 \\
\text{subject to} \quad & \mathbf{1}^\top w = 1 \\
& \mu^\top w \geq r_{\text{target}} \\
& w_i \geq 0 \quad \text{for all } i
\end{aligned}
$$

The $L_2$ penalty discourages concentrated portfolios and improves numerical stability.

In [ ]:
def solve_regularized_target_portfolio(
    mu: np.ndarray,
    cov: np.ndarray,
    target_return: float,
    l2_penalty: float
) -> dict:
    """
    Solve a target-return portfolio with L2 regularisation.
    """
    n = len(mu)

    w = cp.Variable(n)

    risk = cp.quad_form(w, cov)
    regularisation = l2_penalty * cp.sum_squares(w)

    objective = cp.Minimize(risk + regularisation)
    constraints = [
        cp.sum(w) == 1,
        mu @ w >= target_return,
        w >= 0
    ]

    problem = cp.Problem(objective, constraints)
    problem.solve()

    if w.value is None:
        weights = np.full(n, np.nan)
    else:
        weights = np.asarray(w.value).flatten()

    return {
        "status": problem.status,
        "weights": weights,
        "risk": np.nan if w.value is None else float(weights.T @ cov @ weights),
        "objective_value": problem.value,
        "gross_leverage": np.nan if w.value is None else float(np.sum(np.abs(weights))),
        "max_weight": np.nan if w.value is None else float(np.max(weights)),
        "herfindahl_index": np.nan if w.value is None else float(np.sum(weights ** 2))
    }

In [ ]:
target_return = np.quantile(mu, 0.70)

regularisation_rows = []

for l2_penalty in np.logspace(-8, -2, 15):
    result = solve_regularized_target_portfolio(
        mu=mu,
        cov=cov,
        target_return=target_return,
        l2_penalty=l2_penalty
    )

    regularisation_rows.append({
        "l2_penalty": l2_penalty,
        "status": result["status"],
        "risk": result["risk"],
        "max_weight": result["max_weight"],
        "herfindahl_index": result["herfindahl_index"]
    })

regularisation_df = pd.DataFrame(regularisation_rows)
regularisation_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(regularisation_df["l2_penalty"], regularisation_df["herfindahl_index"], marker="o")
plt.xscale("log")
plt.title("L2 Regularisation Reduces Portfolio Concentration")
plt.xlabel(r"L2 penalty $\lambda$")
plt.ylabel("Herfindahl concentration index")
plt.show()

## 8. Transaction-cost-aware rebalancing

Real portfolios are not built from scratch every day. They are rebalanced from an existing portfolio.

Let $w_{\text{old}}$ be the current portfolio and $w$ the new portfolio.

A simple turnover measure is:

$$
\|w - w_{\text{old}}\|_1
$$

A transaction-cost-aware objective can be written as:

$$
\begin{aligned}
&w^\top \Sigma w \\
&\quad + \lambda_{\text{tc}} \|w - w_{\text{old}}\|_1
\end{aligned}
$$

The $L_1$ turnover penalty discourages unnecessary trading.

This is convex and therefore can be solved efficiently.

In [ ]:
def solve_turnover_penalised_portfolio(
    mu: np.ndarray,
    cov: np.ndarray,
    old_weights: np.ndarray,
    target_return: float,
    turnover_penalty: float
) -> dict:
    """
    Solve a target-return portfolio with an L1 turnover penalty.
    """
    n = len(mu)

    w = cp.Variable(n)

    risk = cp.quad_form(w, cov)
    turnover = cp.norm1(w - old_weights)

    objective = cp.Minimize(risk + turnover_penalty * turnover)
    constraints = [
        cp.sum(w) == 1,
        mu @ w >= target_return,
        w >= 0
    ]

    problem = cp.Problem(objective, constraints)
    problem.solve()

    if w.value is None:
        weights = np.full(n, np.nan)
    else:
        weights = np.asarray(w.value).flatten()

    return {
        "status": problem.status,
        "weights": weights,
        "risk": np.nan if w.value is None else float(weights.T @ cov @ weights),
        "turnover": np.nan if w.value is None else float(np.sum(np.abs(weights - old_weights))),
        "achieved_return": np.nan if w.value is None else float(mu @ weights),
        "objective_value": problem.value
    }

In [ ]:
old_weights = np.ones(num_assets) / num_assets
turnover_rows = []

for turnover_penalty in np.logspace(-7, -2, 15):
    result = solve_turnover_penalised_portfolio(
        mu=mu,
        cov=cov,
        old_weights=old_weights,
        target_return=target_return,
        turnover_penalty=turnover_penalty
    )

    turnover_rows.append({
        "turnover_penalty": turnover_penalty,
        "status": result["status"],
        "risk": result["risk"],
        "turnover": result["turnover"],
        "achieved_return": result["achieved_return"]
    })

turnover_df = pd.DataFrame(turnover_rows)
turnover_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(turnover_df["turnover_penalty"], turnover_df["turnover"], marker="o")
plt.xscale("log")
plt.title("Turnover Penalty Reduces Trading")
plt.xlabel(r"Turnover penalty $\lambda_{\mathrm{tc}}$")
plt.ylabel(r"Turnover $\|w - w_{\mathrm{old}}\|_1$")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(turnover_df["turnover"], turnover_df["risk"], marker="o")
plt.title("Risk vs Turnover Trade-Off")
plt.xlabel("Turnover")
plt.ylabel("Portfolio variance")
plt.show()

## 9. Using CVXPY Parameters for fast repeated solves

CVXPY supports parameters.

A parameter is a symbolic value that can be changed without rebuilding the entire optimization problem.

This is useful when sweeping over:

- risk aversion values;
- target returns;
- transaction-cost penalties;
- risk limits;
- scenario assumptions.

Below, we build one problem and repeatedly solve it with different risk-aversion values.

In [ ]:
def risk_return_parameter_sweep(mu: np.ndarray, cov: np.ndarray, gamma_values: np.ndarray) -> pd.DataFrame:
    """
    Solve a long-only mean-variance problem for many risk-aversion values.

    maximise mu^T w - gamma * w^T Sigma w
    """
    n = len(mu)

    w = cp.Variable(n)
    gamma = cp.Parameter(nonneg=True)

    objective = cp.Maximize(mu @ w - gamma * cp.quad_form(w, cov))
    constraints = [
        cp.sum(w) == 1,
        w >= 0
    ]

    problem = cp.Problem(objective, constraints)

    rows = []

    for gamma_value in gamma_values:
        gamma.value = gamma_value
        problem.solve()

        weights = np.asarray(w.value).flatten()

        rows.append({
            "gamma": gamma_value,
            "expected_return": float(mu @ weights),
            "volatility": float(np.sqrt(weights.T @ cov @ weights)),
            "max_weight": float(np.max(weights)),
            "herfindahl_index": float(np.sum(weights ** 2)),
            "status": problem.status
        })

    return pd.DataFrame(rows)


gamma_values = np.logspace(1, 5, 40)
gamma_df = risk_return_parameter_sweep(mu, cov, gamma_values)

gamma_df.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(gamma_df["volatility"], gamma_df["expected_return"], marker="o")
plt.title("Risk-Return Trade-Off from Risk Aversion Sweep")
plt.xlabel("Portfolio volatility")
plt.ylabel("Expected daily return")
plt.show()

## 10. Handling infeasibility

Not every optimization problem has a solution.

For example, in a long-only portfolio, the highest possible expected return is achieved by putting all weight into the asset with the highest expected return.

So if:

$$
r_{\text{target}} > \max_i \mu_i
$$

the target-return problem is infeasible.

A serious optimization workflow should always check the solver status.

In [ ]:
impossible_target = mu.max() + 0.001

impossible_result = solve_target_return_portfolio(
    mu=mu,
    cov=cov,
    target_return=impossible_target
)

print("Impossible target return:", impossible_target)
print("Maximum individual asset expected return:", mu.max())
print("Solver status:", impossible_result["status"])

The solver status is part of the output, not a detail to ignore.

In applied quantitative finance, infeasibility can happen because of:

- unrealistic target returns;
- overly tight risk limits;
- incompatible factor exposure constraints;
- excessive turnover restrictions;
- insufficient eligible assets;
- badly estimated inputs.

## 11. Convex modelling checklist

Before trusting an optimization result, check:

1. **Is the objective convex or concave in the correct direction?**  
   Minimise convex functions or maximise concave functions.

2. **Are the constraints convex?**  
   Equality constraints should usually be affine. Inequality constraints must follow convex optimisation rules.

3. **Is the covariance matrix positive semi-definite?**  
   If not, $w^\top \Sigma w$ may not be convex.

4. **Is the problem feasible?**  
   Always inspect `problem.status`.

5. **Are the weights economically sensible?**  
   Extremely concentrated or unstable weights are warning signs.

6. **Are dual variables interpretable?**  
   Large dual values may reveal which constraints are expensive.

7. **Is the result robust out of sample?**  
   Optimization results are only as good as their inputs.

## 12. Limitations

This notebook deliberately simplifies the optimization problem.

### 12.1 Expected returns are synthetic

In real markets, expected returns are extremely noisy.

Mean-variance optimisation can be unstable because small errors in $\mu$ can cause large changes in weights.

### 12.2 Covariance is assumed known and stable

The covariance matrix is generated synthetically and does not change through time.

Real covariance matrices are time-varying, regime-dependent, noisy, and affected by extreme events.

### 12.3 Transaction costs are simplified

The notebook uses an $L_1$ turnover penalty.

Real transaction costs can include:

- bid-ask spread;
- market impact;
- nonlinear execution cost;
- borrow fees;
- taxes;
- fixed ticket costs;
- exchange fees;
- capacity constraints.

Some of these are convex, while others may introduce non-convexity.

### 12.4 Long-only constraints are not enough

Real portfolios may also need constraints on:

- sectors;
- countries;
- currencies;
- leverage;
- beta;
- duration;
- factor exposures;
- concentration;
- liquidity;
- drawdown risk.

### 12.5 Convexity is powerful but restrictive

Convex optimisation is attractive because global optima can be found reliably.

However, some realistic financial problems are non-convex, especially when they involve:

- integer decisions;
- fixed transaction costs;
- cardinality constraints;
- minimum trade sizes;
- path-dependent objectives;
- discrete execution venues.

In practice, these may require relaxations, heuristics, or mixed-integer optimisation.

## 13. Research frontier and current directions

Convex optimization remains central to quantitative finance, but current research increasingly blends it with machine learning, robust control, and differentiable programming.

### 13.1 Differentiable convex optimization layers

A modern research direction is to embed convex optimization problems inside neural networks.

Instead of a neural network outputting portfolio weights directly, it can output parameters such as expected returns, risk forecasts, or constraints. A convex optimization layer then maps those parameters to feasible portfolio weights.

This gives the model a useful inductive bias: the final output must satisfy hard constraints.

A future notebook could implement a small differentiable mean-variance layer using CVXPYLayers and compare it with an unconstrained neural network.

### 13.2 Robust optimization under estimation error

Classic mean-variance optimization assumes $\mu$ and $\Sigma$ are known.

Robust optimization instead assumes inputs lie in uncertainty sets and seeks portfolios that perform well under adverse but plausible parameter errors.

A future notebook could compare:

- naive mean-variance optimization;
- shrinkage-based optimization;
- robust mean-variance optimization under return uncertainty.

### 13.3 Distributionally robust optimization

Distributionally robust optimization goes beyond uncertainty in parameters.

Instead of trusting one probability distribution, it optimizes against a family of distributions near the empirical distribution.

This is useful for stress testing and tail-risk-aware portfolio construction.

A future notebook could formulate a simple Wasserstein-robust portfolio problem conceptually before moving to implementable approximations.

### 13.4 Convex optimization for optimal execution

Execution problems often trade off risk and market impact.

Some execution models can be written as convex programs, especially when transaction costs and risk terms are convex.

A future notebook could solve a multi-period execution schedule with inventory risk and convex market-impact costs.

### 13.5 Conic optimization and risk measures

Many risk measures can be written using cone constraints.

For example, Conditional Value-at-Risk minimisation can be expressed as a linear or convex optimization problem.

Second-order cone programming and semidefinite programming also appear in robust portfolio construction, covariance uncertainty, and factor exposure control.

A future notebook could build a CVaR optimizer and compare it against variance minimisation.

### 13.6 Mixed-integer extensions

Some practical portfolio constraints are discrete:

- hold at most $K$ assets;
- minimum position size;
- round lots;
- fixed transaction costs.

These constraints create mixed-integer optimization problems.

A future notebook could show how a convex relaxation approximates a cardinality-constrained portfolio and compare it with a mixed-integer solution on a small asset universe.

## 14. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_04_mean_variance_optimization_ledoit_wolf**  
   Applying convex optimization to covariance-shrunk Markowitz portfolios.

2. **04_09_risk_parity_equal_risk_contribution**  
   Studying risk contribution and allocation constraints.

3. **04_11_cvar_convex_optimization**  
   Using convex programming for tail-risk-aware portfolio construction.

4. **04_13_stochastic_control_portfolio_problem**  
   Connecting static optimization to dynamic allocation.

5. **05_05_bayesian_strategy_calibration**  
   Penalised parameter search under turnover and stability constraints.

6. **06_01_almgren_chriss_optimal_execution**  
   Applying convex optimization to execution scheduling.

7. **06_02_dynamic_programming_execution**  
   Comparing convex execution models with dynamic programming approaches.

8. **07_01_multi_asset_cta_research_pipeline**  
   Using optimization as one stage of an end-to-end research pipeline.

## 15. Summary

This notebook introduced convex optimization using CVXPY.

The main workflow was:

1. define a decision variable;
2. define a convex objective;
3. impose realistic constraints;
4. solve the problem;
5. inspect weights, risk, dual values, and solver status.

The key mathematical takeaway is:

> Convex optimization is powerful because it allows realistic constraints while preserving reliable global solutions.

The key financial takeaway is:

> Portfolio construction is not just about finding the mathematically optimal portfolio. It is about finding a feasible, stable, interpretable portfolio under noisy inputs and real-world constraints.

## 16. Further reading

### Core convex optimization

- Boyd, S. and Vandenberghe, L. *Convex Optimization.*
- CVXPY documentation.
- Rockafellar, R. T. *Convex Analysis.*
- Nocedal, J. and Wright, S. *Numerical Optimization.*

### Portfolio optimization

- Markowitz, H. "Portfolio Selection."
- Luenberger, D. G. *Investment Science.*
- Cornuéjols, G. and Tütüncü, R. *Optimization Methods in Finance.*
- Boyd, S. et al. "Multi-Period Trading via Convex Optimization."

### Current directions

- Agrawal, A., Amos, B., Barratt, S., Boyd, S., Diamond, S., and Kolter, Z. "Differentiable Convex Optimization Layers."
- Agrawal, A. and Boyd, S. "Differentiating through Log-Log Convex Programs."
- Busseti, E. *Portfolio Management and Optimal Execution via Convex Optimization.*
- Recent work on distributionally robust optimization, CVXPYLayers, robust portfolio optimization, and differentiable constrained learning.